<a href="https://colab.research.google.com/github/caramos84/GE_AutomationSystem/blob/main/LICITACI%C3%93N_ALDI_Image_Batch_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LICITACIÓN ALDI - Image Batch Downloader
## Descarga masiva desde Excel

Este notebook:

1. Carga un archivo Excel.
2. Lee la columna `Adresy zdjęć`.
3. Extrae las URLs.
4. Descarga las imágenes.
5. Conserva el nombre original de cada archivo cuando el servidor lo entrega.
6. Registra errores y resultados.
7. Genera un ZIP con todas las imágenes descargadas.

La prioridad para determinar el nombre del archivo será:

1. `Content-Disposition` del servidor.
2. Nombre incluido en la URL.
3. Nombre de respaldo únicamente si no existe otra opción.

*   List item
*   List item



In [ ]:
!pip install pandas openpyxl requests tqdm -q

## 1. Importar librerías

In [ ]:
import os
import re
import csv
import shutil
import zipfile
import mimetypes
import requests
import pandas as pd

from urllib.parse import urlparse, unquote
from google.colab import files
from tqdm.auto import tqdm

## 2. Cargar archivo Excel

In [ ]:
uploaded = files.upload()

excel_files = [
    filename
    for filename in uploaded.keys()
    if filename.lower().endswith((".xlsx", ".xls"))
]

if not excel_files:
    raise ValueError("No se cargó ningún archivo Excel.")

EXCEL_PATH = excel_files[0]

print(f"Archivo cargado: {EXCEL_PATH}")

Saving nowy_KW_26_stage_Aktualizacja po druku-historyczny_Aldi_2026-07-10__08_04_22.xlsx to nowy_KW_26_stage_Aktualizacja po druku-historyczny_Aldi_2026-07-10__08_04_22.xlsx
Archivo cargado: nowy_KW_26_stage_Aktualizacja po druku-historyczny_Aldi_2026-07-10__08_04_22.xlsx


## 3. Leer Excel y verificar la columna `Adresy zdjęć`

In [ ]:
df = pd.read_excel(EXCEL_PATH)

TARGET_COLUMN = "Adresy zdjęć"

if TARGET_COLUMN not in df.columns:
    print("Columnas encontradas:")
    print(list(df.columns))
    raise KeyError(
        f"No se encontró la columna '{TARGET_COLUMN}'."
    )

print(f"Columna encontrada: {TARGET_COLUMN}")
print(f"Filas totales: {len(df)}")

Columna encontrada: Adresy zdjęć
Filas totales: 464


## 4. Preparar carpeta de salida

In [ ]:
OUTPUT_DIR = "/content/imagenes_descargadas"

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Carpeta creada: {OUTPUT_DIR}")

Carpeta creada: /content/imagenes_descargadas


## 5. Extraer URLs

Esta celda tolera:

- una URL por celda
- varias URLs separadas por espacios
- saltos de línea
- punto y coma
- comas

In [ ]:
def extract_urls(value):
    if pd.isna(value):
        return []

    text = str(value)

    urls = re.findall(
        r'https?://[^\s,;]+',
        text
    )

    return urls


url_records = []

for index, value in df[TARGET_COLUMN].items():
    urls = extract_urls(value)

    for url in urls:
        url_records.append({
            "row": index + 2,  # +2 por encabezado de Excel
            "url": url
        })

print(f"URLs encontradas: {len(url_records)}")

URLs encontradas: 1330


## 6. Función para detectar el nombre original

Prioridad:

1. `Content-Disposition`
2. Nombre en la URL
3. Nombre de respaldo

In [ ]:
def filename_from_content_disposition(header):
    if not header:
        return None

    # filename*=UTF-8''archivo.jpg
    match = re.search(
        r"filename\*=UTF-8''([^;]+)",
        header,
        flags=re.IGNORECASE
    )

    if match:
        return unquote(match.group(1).strip().strip('"'))

    # filename="archivo.jpg"
    match = re.search(
        r'filename="?([^";]+)"?',
        header,
        flags=re.IGNORECASE
    )

    if match:
        return unquote(match.group(1).strip())

    return None


def filename_from_url(url):
    path = urlparse(url).path
    filename = os.path.basename(path)

    if filename:
        return unquote(filename)

    return None


def infer_extension(response):
    content_type = response.headers.get(
        "Content-Type",
        ""
    ).split(";")[0].strip()

    extension = mimetypes.guess_extension(
        content_type
    )

    return extension or ""

## 7. Evitar sobrescritura accidental

Si dos URLs entregan exactamente el mismo nombre, el notebook conserva ambos archivos agregando un sufijo únicamente para evitar pérdida de datos.

In [ ]:
def make_unique_path(directory, filename):
    base, ext = os.path.splitext(filename)

    candidate = os.path.join(
        directory,
        filename
    )

    counter = 1

    while os.path.exists(candidate):
        candidate = os.path.join(
            directory,
            f"{base}_{counter}{ext}"
        )
        counter += 1

    return candidate

In [ ]:
## 8. Descargar imágenes

In [ ]:
results = []

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "Chrome/120 Safari/537.36"
    )
})


for item in tqdm(
    url_records,
    desc="Descargando imágenes"
):
    url = item["url"]
    row = item["row"]

    try:
        response = session.get(
            url,
            timeout=30,
            allow_redirects=True
        )

        response.raise_for_status()

        # Intento 1: nombre entregado por servidor
        filename = filename_from_content_disposition(
            response.headers.get("Content-Disposition")
        )

        # Intento 2: nombre en URL
        if not filename:
            filename = filename_from_url(
                response.url
            )

        # Intento 3: respaldo
        if not filename:
            extension = infer_extension(
                response
            )

            filename = (
                f"archivo_fila_{row}"
                f"{extension}"
            )

        # Si el nombre no tiene extensión,
        # intentar inferirla desde Content-Type
        if not os.path.splitext(filename)[1]:
            extension = infer_extension(
                response
            )

            filename += extension

        output_path = make_unique_path(
            OUTPUT_DIR,
            filename
        )

        with open(
            output_path,
            "wb"
        ) as f:
            f.write(response.content)

        results.append({
            "row": row,
            "url": url,
            "final_url": response.url,
            "filename": os.path.basename(
                output_path
            ),
            "status": "OK",
            "http_status": response.status_code,
            "error": ""
        })

    except Exception as error:

        results.append({
            "row": row,
            "url": url,
            "final_url": "",
            "filename": "",
            "status": "ERROR",
            "http_status": "",
            "error": str(error)
        })

Descargando imágenes:   0%|          | 0/1330 [00:00<?, ?it/s]

## 9. Generar reporte

In [ ]:
report_df = pd.DataFrame(results)

REPORT_PATH = "/content/download_report.csv"

report_df.to_csv(
    REPORT_PATH,
    index=False,
    encoding="utf-8-sig"
)

display(
    report_df[
        [
            "row",
            "filename",
            "status",
            "http_status"
        ]
    ]
)

print()
print("Resumen:")
print(report_df["status"].value_counts())

,row,filename,status,http_status
0,2,8007708_dallmayr_kawa_rozpuszczalna_01.png,OK,200
1,3,0497-laciate-maslo-ekstra-01.png,OK,200
2,4,8012220-lyttos-grecka-oliwa-z-oliwek-01.png,OK,200
3,4,V_label_vegan.png,OK,200
4,5,9549_sierpc_ser_krolewski_plastry_01.png,OK,200
...,...,...,...,...
1325,465,8020474_6.png,OK,200
1326,465,8020474_7.png,OK,200
1327,465,8020474_8.png,OK,200
1328,465,8020474_9.png,OK,200



Resumen:
status
OK    1330
Name: count, dtype: int64


## 10. Verificar archivos descargados

In [ ]:
downloaded_files = sorted(
    os.listdir(OUTPUT_DIR)
)

print(
    f"Archivos descargados: "
    f"{len(downloaded_files)}"
)

for filename in downloaded_files[:20]:
    print(filename)

if len(downloaded_files) > 20:
    print(
        f"... y {len(downloaded_files) - 20} más"
    )

Archivos descargados: 1330
0196-piatnica-koktajl-bialkowy-01.png
0196-piatnica-koktajl-bialkowy-02.png
0196-piatnica-koktajl-bialkowy-03.png
0196_piatnica_koktajl_odzywczy_mix_01.png
0196_piatnica_koktajl_odzywczy_mix_02.png
0196_piatnica_koktajl_odzywczy_mix_03.png
0277-2.0-PET-SPRITE-03.png
0277-napoje-mix-2l-01.png
0277-napoje-mix-2l-02.png
0300_2.png
0326_cisowianka_niegazowana_01.png
0328-lays-zielona-cebulka-01.png
0406_3.png
0409_4.png
0479-chipsy-lays-fromage-01.png
0497-laciate-maslo-ekstra-01.png
0523_12.png
0559_filet_ze_schabu_wieprzowego_01.png
0605-milsani-mleko-01.png
0605_5.png
... y 1310 más


## 11. Crear ZIP final

El ZIP incluirá:

- todas las imágenes
- `download_report.csv`

In [ ]:
ZIP_PATH = "/content/imagenes_descargadas.zip"

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for filename in os.listdir(
        OUTPUT_DIR
    ):
        filepath = os.path.join(
            OUTPUT_DIR,
            filename
        )

        zipf.write(
            filepath,
            arcname=os.path.join(
                "imagenes",
                filename
            )
        )

    zipf.write(
        REPORT_PATH,
        arcname="download_report.csv"
    )

print(f"ZIP generado: {ZIP_PATH}")

ZIP generado: /content/imagenes_descargadas.zip


## 12. Descargar ZIP

In [ ]:
files.download(
    ZIP_PATH
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>